In [ ]:
# https://my-json-server.typicode.com/st8324/repo/cafe
# 카페 매출 정보를 가져와서 DataFrame으로 변환하는 코드를 작성하세요.
import requests
import pandas as pd

In [ ]:
# 카페 매출 정보를 가져오는 함수
def get_cafe_info():
	url = 'https://my-json-server.typicode.com/st8324/repo/cafe'

	response = requests.get(url)

	try:
		response.raise_for_status()
		return pd.DataFrame(response.json())
	except Exception as e:
		print(f'예외 발생 : {e}')
		return pd.DataFrame()

In [ ]:
# 중복 및 결측치 처리하는 함수
def clean_data(df:pd.DataFrame)->pd.DataFrame:
	# 결측치 제거
	df = df.dropna() 

	return df

In [ ]:
# 확인
df = get_cafe_info()

df = clean_data(df)
# print(df)

In [ ]:
# 데이터를 활용한 예제 
# 매출을 조회하기 위해 '결제액'을 추가
# 결제액은 가격 * 수량
df['결제액'] = df['수량'] * df['가격']
# print(df)


In [ ]:
# 전체 매출액을 조회
revenue = df['결제액'].sum()
print(f'전체 매출액 : {revenue:,}원')

In [ ]:
# 메뉴별 매출액을 조회(메뉴, 수량, 결제액)
# groupby
meun_df = df.groupby('메뉴')[['수량','결제액']].sum()
print(meun_df)

In [ ]:
# 카테고리별 수량과 결제액을 조회
category_df = df.groupby('카테고리')[['수량','결제액']].sum()
# 결제액 기준 내림차순으로 정렬
category_df = category_df.sort_values(by='결제액', ascending=False)

# 카테고리별 매출비율을 추가
# 전체 매출액 : revenue
category_df['매출비율'] = category_df['결제액'] / revenue * 100
category_df['매출비율'] = category_df['매출비율'].round(1) # 소수점 두번째 자리에서 반올림

# 매출이 가장 높은 카테고리를 조회
max_revenue = category_df['결제액'].max()
# print(category_df.loc[category_df['결제액']== max_revenue])

# print("--------")
max_revenue_idx = category_df['결제액'].idxmax() # 결재액 중 가장 큰 값을 가지는 위치
print(f'매출이 가장 많은 카테고리 : {max_revenue_idx}')



In [ ]:
# 문자열 시간을 시간 포맷으로 변환
df['주문시간'] = pd.to_datetime(df['주문시간'])
# 시간대를 추가
df['시간'] = df['주문시간'].dt.hour

# 시간대 별 주문 건수를 조회
hour_df = df['시간'].value_counts()

print(hour_df)

In [ ]:
# 시간대별로 매출액을 조회 : 오전(12시 이전), 오후(17시 이전), 저녁(17시 이후)
